In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np



In [2]:
PROJECT_ROOT = Path.cwd().parent
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"


con = duckdb.connect()

In [3]:
jan_path = INTERIM_DIR / "flights_2025_01.parquet"
print(jan_path.exists())
print(jan_path)


True
/Users/tseringgurung/Desktop/flight-operations-intelligence/data/interim/flights_2025_01.parquet


## Temporal Modeling Strategy

The model will be evaluated using chronological rather than random data splits.

Planned initial structure:

- Training: January–September 2025
- Validation: October–November 2025
- Test: December 2025

This prevents future observations from leaking into model training and better represents real operational deployment.

In [4]:
RAW_FLIGHT_DIR = PROJECT_ROOT / "data" / "raw" / "flights"

FEB_RAW_PATH = RAW_FLIGHT_DIR / "bts_ontime_2025_02.csv"

print(FEB_RAW_PATH.exists())
print(FEB_RAW_PATH)

True
/Users/tseringgurung/Desktop/flight-operations-intelligence/data/raw/flights/bts_ontime_2025_02.csv


In [5]:
con.execute(
    f"""
    CREATE OR REPLACE VIEW flights_feb_raw AS
    SELECT *
    FROM read_csv_auto(
        '{FEB_RAW_PATH}',
        sample_size = 100000,
        ignore_errors = true
    )
    """
)

In [6]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,
    COUNT(DISTINCT FlightDate) AS days
FROM flights_feb_raw
""").df()

,total_rows,first_date,last_date,days
0,504884,2025-02-01,2025-02-28,28


In [7]:
jan_schema = con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{jan_path}')
    """
).df()

feb_schema = con.sql("""
DESCRIBE flights_feb_raw
""").df()

print("January columns:", len(jan_schema))
print("February columns:", len(feb_schema))

January columns: 110
February columns: 110


In [8]:
jan_columns = set(jan_schema["column_name"])
feb_columns = set(feb_schema["column_name"])

print("Missing in February:")
print(jan_columns - feb_columns)

print("\nNew in February:")
print(feb_columns - jan_columns)

Missing in February:
set()

New in February:
set()


In [9]:
same_order = (
    jan_schema["column_name"].tolist()
    == feb_schema["column_name"].tolist()
)

same_order

True

In [10]:
FEB_PARQUET_PATH = (
    PROJECT_ROOT 
    / "Data"
    / "INTERIM"
    /"flights_2025_02.parquet"
)

In [11]:
con.execute(
    f"""
    COPY (
        SELECT *
        FROM flights_feb_raw
    )
    TO '{FEB_PARQUET_PATH}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)

In [12]:
print(FEB_PARQUET_PATH.exists())

con.sql(
    f"""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet('{FEB_PARQUET_PATH}')
    """
).df()

True


,total_rows
0,504884


In [13]:
feb_csv_size_mb = FEB_RAW_PATH.stat().st_size / 1024**2
feb_parquet_size_mb = FEB_PARQUET_PATH.stat().st_size / 1024**2

print(f"CSV size:     {feb_csv_size_mb:.2f} MB")
print(f"Parquet size: {feb_parquet_size_mb:.2f} MB")
print(
    f"Reduction:    "
    f"{(1 - feb_parquet_size_mb / feb_csv_size_mb) * 100:.1f}%"
)

CSV size:     217.67 MB
Parquet size: 12.06 MB
Reduction:    94.5%


In [14]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [15]:
from src.ingest_bts import process_bts_month

In [16]:
feb_result = process_bts_month(
    csv_path=FEB_RAW_PATH,
    parquet_path=FEB_PARQUET_PATH,
    expected_year=2025,
    expected_month=2,
)

feb_result

{'year': 2025,
 'month': 2,
 'rows': 504884,
 'first_date': Timestamp('2025-02-01 00:00:00'),
 'last_date': Timestamp('2025-02-28 00:00:00'),
 'days': 28,
 'csv_size_mb': 217.67,
 'parquet_size_mb': 12.06,
 'storage_reduction_pct': 94.5}

In [17]:
MAR_RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "flights"
    / "bts_ontime_2025_03.csv"
)

MAR_PARQUET_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "flights_2025_03.parquet"
)

print(MAR_RAW_PATH.exists())

True


In [18]:
mar_result = process_bts_month(
    csv_path=MAR_RAW_PATH,
    parquet_path=MAR_PARQUET_PATH,
    expected_year=2025,
    expected_month=3,
)

mar_result

{'year': 2025,
 'month': 3,
 'rows': 600872,
 'first_date': Timestamp('2025-03-01 00:00:00'),
 'last_date': Timestamp('2025-03-31 00:00:00'),
 'days': 31,
 'csv_size_mb': 259.04,
 'parquet_size_mb': 14.58,
 'storage_reduction_pct': 94.4}

In [19]:
mar_result = process_bts_month(
    csv_path=MAR_RAW_PATH,
    parquet_path=MAR_PARQUET_PATH,
    expected_year=2025,
    expected_month=3,
    reference_parquet_path=jan_path,
)

mar_result

{'year': 2025,
 'month': 3,
 'rows': 600872,
 'first_date': Timestamp('2025-03-01 00:00:00'),
 'last_date': Timestamp('2025-03-31 00:00:00'),
 'days': 31,
 'csv_size_mb': 259.04,
 'parquet_size_mb': 14.58,
 'storage_reduction_pct': 94.4}

In [20]:
RAW_FLIGHT_DIR = PROJECT_ROOT / "data" / "raw" / "flights"

for month in range(1, 13):
    path = RAW_FLIGHT_DIR / f"bts_ontime_2025_{month:02d}.csv"
    print(f"{month:02d}: {path.exists()}")

01: True
02: True
03: True
04: True
05: True
06: True
07: True
08: True
09: True
10: True
11: True
12: True


In [21]:
ingestion_results = []

for month in range(4, 13):

    csv_path = (
        RAW_FLIGHT_DIR
        / f"bts_ontime_2025_{month:02d}.csv"
    )

    parquet_path = (
        INTERIM_DIR
        / f"flights_2025_{month:02d}.parquet"
    )

    print(f"Processing 2025-{month:02d}...")

    result = process_bts_month(
        csv_path=csv_path,
        parquet_path=parquet_path,
        expected_year=2025,
        expected_month=month,
        reference_parquet_path=jan_path,
    )

    ingestion_results.append(result)

    print(
        f"✓ {result['rows']:,} rows | "
        f"{result['storage_reduction_pct']}% reduction"
    )

Processing 2025-04...
✓ 583,950 rows | 94.5% reduction
Processing 2025-05...
✓ 605,648 rows | 94.5% reduction
Processing 2025-06...
✓ 611,575 rows | 94.5% reduction
Processing 2025-07...
✓ 631,428 rows | 94.4% reduction
Processing 2025-08...
✓ 602,378 rows | 94.4% reduction
Processing 2025-09...
✓ 562,439 rows | 94.6% reduction
Processing 2025-10...
✓ 605,844 rows | 94.5% reduction
Processing 2025-11...
✓ 570,550 rows | 94.4% reduction
Processing 2025-12...
✓ 582,304 rows | 94.1% reduction


In [22]:
ingestion_summary = pd.DataFrame(ingestion_results)

ingestion_summary

,year,month,rows,first_date,last_date,days,csv_size_mb,parquet_size_mb,storage_reduction_pct
0,2025,4,583950,2025-04-01,2025-04-30,30,251.85,13.96,94.5
1,2025,5,605648,2025-05-01,2025-05-31,31,261.62,14.33,94.5
2,2025,6,611575,2025-06-01,2025-06-30,30,264.55,14.64,94.5
3,2025,7,631428,2025-07-01,2025-07-31,31,272.92,15.21,94.4
4,2025,8,602378,2025-08-01,2025-08-31,31,260.06,14.49,94.4
5,2025,9,562439,2025-09-01,2025-09-30,30,242.30,13.14,94.6
6,2025,10,605844,2025-10-01,2025-10-31,31,261.98,14.42,94.5
7,2025,11,570550,2025-11-01,2025-11-30,30,246.12,13.84,94.4
8,2025,12,582304,2025-12-01,2025-12-31,31,252.27,14.90,94.1


In [23]:
all_flights_path = INTERIM_DIR / "flights_2025_*.parquet"

con.execute(
    f"""
    CREATE OR REPLACE VIEW flights_2025 AS
    SELECT *
    FROM read_parquet('{all_flights_path}')
    """
)

In [24]:
year_validation = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,
    COUNT(DISTINCT FlightDate) AS days,
    COUNT(DISTINCT Month) AS months
FROM flights_2025
""").df()

year_validation

,total_rows,first_date,last_date,days,months
0,7001619,2025-01-01,2025-12-31,365,12


In [25]:
monthly_counts = con.sql("""
SELECT
    Month,
    COUNT(*) AS flights
FROM flights_2025
GROUP BY Month
ORDER BY Month
""").df()

monthly_counts

,Month,flights
0,1,539747
1,2,504884
2,3,600872
3,4,583950
4,5,605648
5,6,611575
6,7,631428
7,8,602378
8,9,562439
9,10,605844


In [26]:
modeling_2025 = con.sql("""
SELECT
    Year,
    Quarter,
    Month,
    DayofMonth,
    DayOfWeek,
    FlightDate,

    Reporting_Airline,
    Origin,
    Dest,

    CRSDepTime,
    CRSArrTime,
    CRSElapsedTime,

    Distance,
    DistanceGroup,

    Origin || '_' || Dest AS route,

    CAST(
        FLOOR(CAST(CRSDepTime AS INTEGER) / 100)
        AS INTEGER
    ) AS scheduled_dep_hour,

    CAST(
        FLOOR(CAST(CRSArrTime AS INTEGER) / 100)
        AS INTEGER
    ) AS scheduled_arr_hour,

    CASE
        WHEN DayOfWeek IN (6, 7) THEN 1
        ELSE 0
    END AS is_weekend,

    CASE
        WHEN CAST(CRSDepTime AS INTEGER) < 600 THEN 'overnight'
        WHEN CAST(CRSDepTime AS INTEGER) < 1200 THEN 'morning'
        WHEN CAST(CRSDepTime AS INTEGER) < 1800 THEN 'afternoon'
        ELSE 'evening'
    END AS departure_period,

    ArrDelay,

    CASE
        WHEN ArrDelay >= 15 THEN 1
        ELSE 0
    END AS significant_arrival_delay

FROM flights_2025

WHERE Cancelled = 0
  AND Diverted = 0
  AND ArrDelay IS NOT NULL
""")

In [27]:
modeling_2025.aggregate("""
    COUNT(*) AS modeling_rows
""").df()

,modeling_rows
0,6879484


In [28]:
year_target = con.sql("""
SELECT
    significant_arrival_delay,
    COUNT(*) AS flights,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM modeling_2025
GROUP BY significant_arrival_delay
ORDER BY significant_arrival_delay
""").df()

year_target

,significant_arrival_delay,flights,percentage
0,0,5344846,77.69
1,1,1534638,22.31


In [29]:
monthly_target = con.sql("""
SELECT
    Month,
    COUNT(*) AS flights,
    SUM(significant_arrival_delay) AS delayed_flights,
    ROUND(
        AVG(significant_arrival_delay) * 100,
        2
    ) AS delay_rate_pct
FROM modeling_2025
GROUP BY Month
ORDER BY Month
""").df()

monthly_target

,Month,flights,delayed_flights,delay_rate_pct
0,1,522269,98130.0,18.79
1,2,496476,103102.0,20.77
2,3,592301,116034.0,19.59
3,4,577730,113604.0,19.66
4,5,597574,141038.0,23.60
5,6,599472,169405.0,28.26
6,7,612811,177012.0,28.89
7,8,593733,133920.0,22.56
8,9,558328,92859.0,16.63
9,10,601570,122251.0,20.32


In [30]:
train_2025 = con.sql("""
SELECT *
FROM modeling_2025
WHERE Month BETWEEN 1 AND 9
""")

validation_2025 = con.sql("""
SELECT *
FROM modeling_2025
WHERE Month BETWEEN 10 AND 11
""")

test_2025 = con.sql("""
SELECT *
FROM modeling_2025
WHERE Month = 12
""")

In [31]:
split_summary = con.sql("""
SELECT
    'train' AS split,
    COUNT(*) AS flights,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,
    ROUND(AVG(significant_arrival_delay) * 100, 2) AS delay_rate_pct
FROM train_2025

UNION ALL

SELECT
    'validation',
    COUNT(*),
    MIN(FlightDate),
    MAX(FlightDate),
    ROUND(AVG(significant_arrival_delay) * 100, 2)
FROM validation_2025

UNION ALL

SELECT
    'test',
    COUNT(*),
    MIN(FlightDate),
    MAX(FlightDate),
    ROUND(AVG(significant_arrival_delay) * 100, 2)
FROM test_2025
""").df()

split_summary

,split,flights,first_date,last_date,delay_rate_pct
0,train,5150694,2025-01-01,2025-09-30,22.23
1,validation,1156866,2025-10-01,2025-11-30,20.44
2,test,571924,2025-12-01,2025-12-31,26.77
